# Sports14 Text/Image Feature Extraction

In [2]:

import os
import numpy as np
import pandas as pd

In [3]:
os.chdir('/home/minhle/CaMRec/data')
os.getcwd()

'/home/minhle/CaMRec/data'

## Load text data

In [4]:
i_id, desc_str = 'itemID', 'description'

file_path = './'
file_name = 'meta-baby.csv'

meta_file = os.path.join(file_path, file_name)

df = pd.read_csv(meta_file)
df.sort_values(by=[i_id], inplace=True)

print('data loaded!')
print(f'shape: {df.shape}')

df[:3]

data loaded!
shape: (7050, 10)


,itemID,asin,categories,description,title,price,imUrl,brand,related,salesRank
0,0,097293751X,[['Baby']],Easily keep track of your baby's or child's da...,"Baby Tracker&reg; - Daily Childcare Journal, S...",17.00,http://ecx.images-amazon.com/images/I/41Bb6wf%...,Time Too,"{'also_bought': ['9729375011', 'B004FN1AE8', '...",NaN
1,1,9729375011,[['Baby']],This is version of the award-winningBaby Track...,Newborn Baby Tracker&reg; - Round the Clock Ch...,15.95,http://ecx.images-amazon.com/images/I/51r3BLpL...,NaN,"{'also_bought': ['B000V5KPZ4', 'B001F8TLLU', '...",NaN
2,2,B00000IZQI,[['Baby']],This colorful car collection develops motor sk...,Fisher Price Nesting Action Vehicles,8.37,http://ecx.images-amazon.com/images/I/51E83QCC...,NaN,"{'also_bought': ['B0042D69W4', 'B00428LIZM', '...",NaN


In [5]:

# sentences: title + brand + category + description | All have title + description

title_na_df = df[df['title'].isnull()]
print(title_na_df.shape)

desc_na_df = df[df['description'].isnull()]
print(desc_na_df.shape)

na_df = df[df['description'].isnull() & df['title'].isnull()]
print(na_df.shape)

na3_df = df[df['description'].isnull() & df['title'].isnull() & df['brand'].isnull()]
print(na3_df.shape)

na4_df = df[df['description'].isnull() & df['title'].isnull() & df['brand'].isnull() & df['categories'].isnull()]
print(na4_df.shape)

(13, 10)
(664, 10)
(13, 10)
(13, 10)
(0, 10)


In [6]:

df[desc_str] = df[desc_str].fillna(" ")
df['title'] = df['title'].fillna(" ")
df['brand'] = df['brand'].fillna(" ")
df['categories'] = df['categories'].fillna(" ")


In [7]:
sentences = []
for i, row in df.iterrows():
    sen = row['title'] + ' ' + row['brand'] + ' '
    cates = eval(row['categories'])
    if isinstance(cates, list):
        for c in cates[0]:
            sen = sen + c + ' '
    sen += row[desc_str]
    sen = sen.replace('\n', ' ')

    sentences.append(sen)

sentences[:10]

["Baby Tracker&reg; - Daily Childcare Journal, Schedule Log Time Too Baby Easily keep track of your baby's or child's daily schedules, activities and needs in one handy spot.  And make the drop-off and pick-up process for childcare a snap.  The Baby Tracker Daily Childcare Journal is designed to make it fast and easy to record and review meals, naps, activities, playtime, daily news, milestones and to-dos ---all in a single page view.  Get ready for doctor visits, give to childcare helpers and use this easy at-a-glance daily record to monitor schedules and prep for your baby's daily needs.  This version records daily activities from 6 a.m. to 7 p.m. and creates a visual snapshot of activities by time each day.   The Baby Tracker Daily Childcare Journal is ready to keep track of up to 6 months of daily schedule tracking (180 days) and is spiral bound to make it portable and lay flat for easy recording on a countertop or nightstand.Each journal includes:*180 daily at-a-glance tracking re

In [8]:

course_list = df[i_id].tolist()
#sentences = df[desc_str].tolist()

assert course_list[-1] == len(course_list) - 1

In [9]:
# should `pip install sentence_transformers` first
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

sentence_embeddings = model.encode(sentences)
print('text encoded!')

assert sentence_embeddings.shape[0] == df.shape[0]
np.save(os.path.join(file_path, 'text_feat.npy'), sentence_embeddings)
print('done!')


/home/minhle/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


text encoded!
done!


In [10]:
sentence_embeddings[:10]

array([[-0.07651239, -0.00465495, -0.04175242, ..., -0.02215183,
         0.01199163,  0.03745584],
       [-0.074025  , -0.02081848, -0.0869366 , ...,  0.01936686,
         0.03331775,  0.00943885],
       [-0.02278868,  0.0234328 ,  0.0098731 , ...,  0.04196899,
         0.02564454,  0.10332294],
       ...,
       [-0.04872961,  0.04490934, -0.00986575, ..., -0.0261429 ,
         0.08081724,  0.08468328],
       [-0.00700678, -0.009185  , -0.00793412, ..., -0.00388827,
        -0.02889496,  0.08256401],
       [-0.054946  ,  0.01991389, -0.03054535, ..., -0.03612386,
         0.06067673, -0.03718201]], shape=(10, 384), dtype=float32)

In [11]:
load_txt_feat = np.load('text_feat.npy', allow_pickle=True)
print(load_txt_feat.shape)
load_txt_feat[:10]

(7050, 384)


array([[-0.07651239, -0.00465495, -0.04175242, ..., -0.02215183,
         0.01199163,  0.03745584],
       [-0.074025  , -0.02081848, -0.0869366 , ...,  0.01936686,
         0.03331775,  0.00943885],
       [-0.02278868,  0.0234328 ,  0.0098731 , ...,  0.04196899,
         0.02564454,  0.10332294],
       ...,
       [-0.04872961,  0.04490934, -0.00986575, ..., -0.0261429 ,
         0.08081724,  0.08468328],
       [-0.00700678, -0.009185  , -0.00793412, ..., -0.00388827,
        -0.02889496,  0.08256401],
       [-0.054946  ,  0.01991389, -0.03054535, ..., -0.03612386,
         0.06067673, -0.03718201]], shape=(10, 384), dtype=float32)

# Image encoder (V0)，following LATTICE, averaging over for missed items

In [12]:
df[:5]

,itemID,asin,categories,description,title,price,imUrl,brand,related,salesRank
0,0,097293751X,[['Baby']],Easily keep track of your baby's or child's da...,"Baby Tracker&reg; - Daily Childcare Journal, S...",17.00,http://ecx.images-amazon.com/images/I/41Bb6wf%...,Time Too,"{'also_bought': ['9729375011', 'B004FN1AE8', '...",NaN
1,1,9729375011,[['Baby']],This is version of the award-winningBaby Track...,Newborn Baby Tracker&reg; - Round the Clock Ch...,15.95,http://ecx.images-amazon.com/images/I/51r3BLpL...,,"{'also_bought': ['B000V5KPZ4', 'B001F8TLLU', '...",NaN
2,2,B00000IZQI,[['Baby']],This colorful car collection develops motor sk...,Fisher Price Nesting Action Vehicles,8.37,http://ecx.images-amazon.com/images/I/51E83QCC...,,"{'also_bought': ['B0042D69W4', 'B00428LIZM', '...",NaN
3,3,B00000J3LL,[['Baby']],This darling cloth book offers hands-on experi...,"My Quiet Book, Fabric Activity Book for Children",27.00,http://ecx.images-amazon.com/images/I/51GoNXhB...,,"{'also_bought': ['B00000J3LC', 'B0043G4JOA', '...",NaN
4,4,B00002JV9S,[['Baby']],"In a relatively new concept in teething, The F...",The First Years Massaging Action Teether,8.84,http://ecx.images-amazon.com/images/I/41gVp98n...,The First Years,"{'also_bought': ['B0013FCBJO', 'B0019QCGVK', '...",NaN


In [13]:
import array

def readImageFeatures(path):
  f = open(path, 'rb')
  while True:
    asin = f.read(10).decode('UTF-8')
    if asin == '': break
    a = array.array('f')
    a.fromfile(f, 4096)
    yield asin, a.tolist()

In [18]:

img_data = readImageFeatures(r"../raw_data/image_features_Baby.b")
item2id = dict(zip(df['asin'], df['itemID']))

feats = {}
avg = []
for d in img_data:
    if d[0] in item2id:
        feats[int(item2id[d[0]])] = d[1]
        avg.append(d[1])
avg = np.array(avg).mean(0).tolist()

ret = []
non_no = []
for i in range(len(item2id)):
    if i in feats:
        ret.append(feats[i])
    else:
        non_no.append(i)
        ret.append(avg)

print('# of items not in processed image features:', len(non_no))
assert len(ret) == len(item2id)
np.save('image_feat.npy', np.array(ret))
np.savetxt("missed_img_itemIDs.csv", non_no, delimiter =",", fmt ='%d')
print('done!')

# of items not in processed image features: 63
done!
